In [16]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch

In [17]:
housing = fetch_california_housing()

In [18]:

X_train, X_test, y_train, y_test = train_test_split(housing.data, housing.target, random_state=42)

In [19]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, random_state=42)

normalize the data, using tensor operations instead of `StandardScaler`

In [ ]:

X_train = torch.FloatTensor(X_train) # FloatTensor turns the 64bit numpy array X_train to a 32bit tensor (instead of a 62 bit)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)

means = X_train.mean(dim=0, keepdims=True) # average of the columns, collapsing the row. gives us the mean per feature
stds = X_train.std(dim=0, keepdims=True) # same


# normalizing step
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

we also need to convert our targets to tensors

since our preds will be column vectors, we need to ensure our targets are also columns vectors

originally its a 1d array of prices, so we need to reshape to column vectors (-1 is a wildcard for rows, 1 col per row)

In [ ]:
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

In [31]:
# parameters (rand weights instead of 0 to break symmetry)
torch.manual_seed(42)
n_features = X_train.shape[1] # 8 input features
w = torch.randn((n_features, 1), requires_grad=True) # weights of each feature, 8x1 tensor of random values (column vector)
b = torch.tensor(0., requires_grad=True) # bias param

this is a single layer nn, so 1 set of weights for the inputs, and 1 output neuron  matmul the input features w the weights and add bias to get our output, then get loss and perform gradient descent (batch gd in this case)

In [32]:
learning_rate = 0.4
n_epochs = 100

for epoch in range(n_epochs):
    y_pred = X_train @ w + b  # (whatever, 1) vector of predictions matching w/ the size of y_train
    loss = ((y_pred - y_train) ** 2).mean() # MSE
    loss.backward() # gradients of the loss w/ regards to each of the 8 input features and the bias 
    with torch.no_grad():
        print(b.grad)
        print(w.grad)
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad # w.grad is 8,1 matrix
        b.grad.zero_()
        w.grad.zero_()
    print(f'Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}')
    

tensor(-4.1643)
tensor([[-1.1391],
        [ 0.6704],
        [ 1.3922],
        [ 1.3897],
        [-2.9650],
        [-0.6495],
        [ 6.2128],
        [-5.5244]])
Epoch 1/100, Loss: 16.158458709716797
tensor(-0.8328)
tensor([[-0.1458],
        [-0.9254],
        [-1.1661],
        [-1.1926],
        [ 0.7602],
        [ 0.5226],
        [-3.3911],
        [ 3.7948]])
Epoch 2/100, Loss: 4.87936544418335
tensor(-0.1666)
tensor([[-0.0317],
        [ 0.0908],
        [ 0.9437],
        [ 0.7035],
        [-0.8431],
        [-0.2203],
        [ 2.3798],
        [-1.9177]])
Epoch 3/100, Loss: 2.255225896835327
tensor(-0.0333)
tensor([[-0.0845],
        [-0.1974],
        [-0.5820],
        [-0.6768],
        [ 0.3245],
        [ 0.1860],
        [-1.1485],
        [ 1.4693]])
Epoch 4/100, Loss: 1.3307628631591797
tensor(-0.0067)
tensor([[ 0.0289],
        [ 0.0403],
        [ 0.5006],
        [ 0.3197],
        [-0.2869],
        [-0.0921],
        [ 0.9783],
        [-0.6102]])
Epoch 

In [38]:
X_new = X_test[:3] # pretend like this is new

with torch.no_grad(): # doenst keep track of computation graph so faster
    y_pred = X_new @ w + b # use trained param to make preds

print(y_pred)
print(y_test[:3])
print(y_pred - y_test[:3])

tensor([[0.7090],
        [1.7824],
        [2.7693]])
tensor([[0.4770],
        [0.4580],
        [5.0000]])
tensor([[ 0.2320],
        [ 1.3244],
        [-2.2307]])


## Rewrite using pytorchs higher level api

In [ ]:
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1) # 8 inputs, 1 neuron (output neuron)

In [40]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [ ]:
model.weight # only 1 row, since 1 neuron, 1 row per neuron

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [ ]:
for param in model.parameters(): # weights and bias
    print(param)

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)
Parameter containing:
tensor([0.3117], requires_grad=True)


In [ ]:
model(X_train[:2]) # not trained yet, preds terrible
# model internall calls forward() method (X_train[:2] @ self.weight.T + self.bias)

tensor([[-0.4718],
        [ 0.1131]], grad_fn=<AddmmBackward0>)

create an optimizer to update model params, and choose a loss function
- optimizer: stochastic gradient descent (can be used for sgd, mini batch gd, or batch gd)
- loss function: mean squared error (mse)

optimizer is the object that actually updates the model params using the gradients that autograd created
- give it model.parameters(), an iterator over all parameters in the model so it knows what to update
- learning rate, for how much to update based on the .grad of each param

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)  
mse = nn.MSELoss()

In [ ]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train) # call, loss function (i.e. mse in this case), `loss` is a tensor w/ .grad_fn
        loss.backward() # compute gradients and store them in each params .grad. 
                        # loss has computation graph, with y_pred, so it can walk backwards to orignal w and b
        optimizer.step() # take the gradients and decide how to apply them (update w and b)
        optimizer.zero_grad()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [49]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/100, Loss: 4.3378496170043945
Epoch 2/100, Loss: 0.7802932858467102
Epoch 3/100, Loss: 0.6253840923309326
Epoch 4/100, Loss: 0.6060433983802795
Epoch 5/100, Loss: 0.5956298112869263
Epoch 6/100, Loss: 0.5873566269874573
Epoch 7/100, Loss: 0.5802990198135376
Epoch 8/100, Loss: 0.5741382241249084
Epoch 9/100, Loss: 0.5687100291252136
Epoch 10/100, Loss: 0.5639079213142395
Epoch 11/100, Loss: 0.5596511363983154
Epoch 12/100, Loss: 0.555873692035675
Epoch 13/100, Loss: 0.5525194406509399
Epoch 14/100, Loss: 0.5495391488075256
Epoch 15/100, Loss: 0.5468899011611938
Epoch 16/100, Loss: 0.544533908367157
Epoch 17/100, Loss: 0.5424376726150513
Epoch 18/100, Loss: 0.5405715703964233
Epoch 19/100, Loss: 0.5389097332954407
Epoch 20/100, Loss: 0.5374287962913513
Epoch 21/100, Loss: 0.536108672618866
Epoch 22/100, Loss: 0.5349311232566833
Epoch 23/100, Loss: 0.5338802933692932
Epoch 24/100, Loss: 0.5329418182373047
Epoch 25/100, Loss: 0.532103419303894
Epoch 26/100, Loss: 0.5313538908958435

In [ ]:
X_new = X_test[:3]
with torch.no_grad():
    y_pred = model(X_new)

y_pred # similar to prev, but not same b/c nn.Linear init parms slightly diff 

tensor([[0.7070],
        [1.7841],
        [2.7667]])